# Prétraitement et Nettoyage des Données : Titanic pour un SVM

Ce notebook présente étape par étape le **nettoyage et la préparation** (*Preprocessing*) du jeu de données Titanic afin qu'il puisse être utilisé par un algorithme **SVM** (*Support Vector Machine*).


## Rappel pédagogique : Pourquoi un SVM a-t-il besoin d'un prétraitement strict ?

Un modèle SVM cherche à séparer les passagers en deux groupes (les survivants `1` et les victimes `0`) en traçant une frontière mathématique (un *hyperplan*) qui maximise la distance avec les points les plus proches. Pour calculer ces distances géométriques, le SVM impose trois contraintes absolues :

1. **Aucune valeur manquante (`NaN`)** : Une formule mathématique de distance ne peut pas additionner ou soustraire une case vide. Toutes les valeurs manquantes doivent être remplacées (*imputées*).
2. **Uniquement des nombres** : Un SVM ne sait pas calculer la distance entre `"male"` et `"female"` ou entre `"Southampton"` et `"Cherbourg"`. Toutes les catégories doivent être converties en valeurs numériques (*encodées*).
3. **La même échelle pour toutes les variables numériques** : Si le prix du billet (`Fare`) varie de `0` à `512` et que le nombre de conjoints/frères/sœurs (`SibSp`) varie de `0` à `8`, la distance géométrique sera presque entièrement déterminée par le prix du billet. Il faut donc ramener toutes les variables numériques à une échelle comparable (*standardisation*).



## Plan du prétraitement

1. **Chargement des données**
2. **Séparation de la cible ($y$) et sélection des variables explicatives ($X$)** (suppression des fuites de données et des colonnes non pertinentes)
3. **Découpage en deux ensembles : Entraînement (`Train`) et Test (`Test`)** avec stratification
4. **Traitement des variables numériques standards** (`Age`, `SibSp`, `Parch`) : Imputation par la médiane et Standardisation
5. **Traitement spécifique de la variable `Fare`** : Imputation par la médiane, Transformation logarithmique (`log1p`) et Standardisation
6. **Traitement de la variable binaire `Sex`** : Imputation par la valeur la plus fréquente et Encodage ordinal (`0` / `1`)
7. **Traitement des variables catégorielles `Pclass` et `Embarked`** : Imputation par la valeur la plus fréquente et Encodage One-Hot (`drop='first'`)
8. **Assemblage final avec `ColumnTransformer` et `Pipeline`**
9. **Application du pipeline (`fit_transform` sur `Train`, `transform` sur `Test`) et inspection du résultat final**
10. **Exportation des ensembles `Train` et `Test` préparés au format CSV dans le dossier `dataset/`**


## Étape 1 : Importation des outils et chargement du jeu de données

Nous importons `pandas` et `numpy` pour manipuler les tableaux de données, ainsi que les briques de prétraitement de la bibliothèque `scikit-learn` (`sklearn`).


In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# Chargement du fichier CSV
df = pd.read_csv("dataset/titanic.csv")

print(f"Dimensions initiales du dataset : {df.shape[0]} lignes et {df.shape[1]} colonnes.")
display(df.head(3))


Dimensions initiales du dataset : 1309 lignes et 15 colonnes.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,NaN,NaN,"Bridgerule, Devon"
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,4,NaN,"New York, NY"
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,NaN,NaN,NaN


---
## Étape 2 : Séparation de la cible ($y$) et nettoyage des colonnes ($X$)

Conformément aux conclusions de notre Analyse Exploratoire (EDA), nous devons séparer notre tableau en deux éléments :
- **La cible $y$ (`Survived`)** : La colonne que le modèle devra apprendre à prédire (`0` = Décédé, `1` = Survivant).
- **Les variables explicatives $X$** : Les colonnes utiles pour prédire la survie.

Pour construire $X$, nous retirons :
1. **`Survived`** : Car c'est la réponse à deviner (elle va dans $y$).
2. **`boat` et `body`** : Ce sont des **fuites de données (*Data Leakage*)**. Elles indiquent le canot de sauvetage ou le numéro du corps repêché *après* le naufrage. Les conserver reviendrait à donner la réponse au modèle.
3. **`PassengerId`** : Simple numéro de ligne séquentiel (`1` à `1309`) sans aucun lien avec la survie.
4. **`Name` et `Ticket`** : Textes à très haute cardinalité (presque une valeur unique par passager), inutilisables tels quels par un SVM.
5. **`Cabin` et `home.dest`** : Colonnes textuelles comportant trop de valeurs manquantes (respectivement 77,5 % et 43,1 % de valeurs manquantes).


In [9]:
# Liste des colonnes à exclure des variables explicatives X
colonnes_a_supprimer = [
    "Survived",     # Variable cible (placée dans y)
    "boat",         # Fuite de données (Data Leakage)
    "body",         # Fuite de données (Data Leakage)
    "PassengerId",  # Identifiant unique sans valeur prédictive
    "Name",         # Texte brut à haute cardinalité
    "Ticket",       # Code alphanumérique à haute cardinalité
    "Cabin",        # 77.5 % de valeurs manquantes
    "home.dest"     # 43.1 % de valeurs manquantes et texte libre
]

# Création de X (variables explicatives) et y (variable cible)
X = df.drop(columns=colonnes_a_supprimer)
y = df["Survived"]

print("Colonnes conservées dans X :", list(X.columns))
print(f"Dimensions de X : {X.shape}")
print(f"Dimensions de y : {y.shape}")
display(X.head(5))


Colonnes conservées dans X : ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Dimensions de X : (1309, 7)
Dimensions de y : (1309,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


## Étape 3 : Découpage en deux ensembles (`Train` et `Test`)

### Pourquoi découper maintenant, avant de calculer les médianes ou de standardiser ?

C'est la règle la plus importante du prétraitement en Machine Learning :
- Le **jeu de test (`X_test`, `y_test`)** représente de nouveaux passagers inconnus sur lesquels on évaluera le modèle à la toute fin.
- Si l'on calculait l'âge médian ou la moyenne des prix sur les 1 309 passagers *avant* de découper, de l'information issue du jeu de test glisserait dans le jeu d'entraînement (c'est une forme subtile de *Data Leakage*).
- En découpant **dès maintenant**, nos outils de nettoyage apprendront les médianes et les moyennes **uniquement sur `X_train`**, puis appliqueront ces mêmes règles à `X_test`.

### Les paramètres de `train_test_split` expliqués :
- **`test_size=0.2`** : Nous réservons **20 %** des données pour le test final (262 passagers) et **80 %** pour l'entraînement (1 047 passagers).
- **`random_state=42`** : Fixe la graine du générateur aléatoire. Cela garantit qu'à chaque exécution du notebook, exactement les mêmes passagers iront dans `Train` et dans `Test` (reproductibilité scientifique).
- **`stratify=y`** : Force le découpage aléatoire à conserver exactement la même proportion de survivants (**38,2 %**) et de décédés (**61,8 %**) dans le `Train` et dans le `Test`.


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Ensemble d'entraînement (Train) : X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Ensemble de test        (Test)  : X_test  = {X_test.shape},  y_test  = {y_test.shape}")

print("\nVérification de la stratification (proportion de survivants) :")
print(f" - Dans y_train : {y_train.mean() * 100:.2f} %")
print(f" - Dans y_test  : {y_test.mean() * 100:.2f} %")


Ensemble d'entraînement (Train) : X_train = (1047, 7), y_train = (1047,)
Ensemble de test        (Test)  : X_test  = (262, 7),  y_test  = (262,)

Vérification de la stratification (proportion de survivants) :
 - Dans y_train : 38.20 %
 - Dans y_test  : 38.17 %


## Étape 4 : Pipeline pour les variables numériques standards (`Age`, `SibSp`, `Parch`)

Ces trois variables sont des nombres :
- **`Age`** : Âge du passager (de 0,17 à 80 ans, avec ~20 % de valeurs manquantes).
- **`SibSp`** : Nombre de frères, sœurs, époux ou épouses à bord (de 0 à 8).
- **`Parch`** : Nombre de parents ou d'enfants à bord (de 0 à 9).

Nous leur appliquons deux opérations successives regroupées dans un `Pipeline` :
1. **`SimpleImputer(strategy='median')`** : Remplace toute valeur manquante (`NaN`) par la **médiane** de la colonne (calculée sur `X_train`). On préfère la médiane à la moyenne car elle n'est pas faussée par les valeurs extrêmes.
2. **`StandardScaler()`** : Standardise chaque colonne selon la formule :
   $$z = \frac{x - \mu}{\sigma}$$
   où $\mu$ est la moyenne et $\sigma$ l'écart-type (calculés sur `X_train`). Après cette opération, `Age`, `SibSp` et `Parch` auront toutes une **moyenne de 0** et un **écart-type de 1**, ce qui est indispensable pour le calcul des distances du SVM.


In [11]:
colonnes_numeriques_normales = ["Age", "SibSp", "Parch"]

pipeline_numerique = Pipeline([
    ("imputation", SimpleImputer(strategy="median")),
    ("standardisation", StandardScaler())
])

print("Pipeline numérique standard configuré pour :", colonnes_numeriques_normales)


Pipeline numérique standard configuré pour : ['Age', 'SibSp', 'Parch']


## Étape 5 : Pipeline spécifique pour la variable `Fare` (Prix du billet)

Lors de l'EDA, nous avons découvert que `Fare` présente deux particularités :
1. Elle comporte une valeur manquante (`NaN`).
2. Sa distribution est **extrêmement asymétrique à droite** (*Skewness* = 4,37) : la majorité des billets coûte entre 7 £ et 31 £ (médiane à 14,45 £), mais quelques billets atteignent **512,33 £**.

Si l'on appliquait directement `StandardScaler()` sur `Fare`, ces valeurs à 512 £ resteraient des points très éloignés (*outliers*) qui déformeraient la frontière du SVM.

Nous construisons donc un pipeline en **3 étapes** pour `Fare` :

1. **`SimpleImputer(strategy='median')`** : Remplace toute valeur manquante par la médiane.

2. **`FunctionTransformer(np.log1p, feature_names_out='one-to-one')`** : Applique la fonction mathématique $\ln(1 + x)$ à chaque prix.
   - *Pourquoi $\ln(1 + x)$ et pas $\ln(x)$ ?* Parce que certains billets gratuits coûtent `0.00 £`. Or $\ln(0)$ est impossible ($-\infty$), tandis que $\ln(1 + 0) = \ln(1) = 0$.
   - Cette transformation compresse les grands écarts (512 £ devient $\ln(513) \approx 6,24$ tandis que 14,45 £ devient $\ln(15,45) \approx 2,74$), ramenant l'asymétrie près de 0.
   
3. **`StandardScaler()`** : Centre et réduit la variable transformée (moyenne = 0, écart-type = 1).


In [12]:
colonne_fare = ["Fare"]

pipeline_fare = Pipeline([
    ("imputation", SimpleImputer(strategy="median")),
    ("logarithme", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("standardisation", StandardScaler())
])

print("Pipeline spécifique configuré pour :", colonne_fare)


Pipeline spécifique configuré pour : ['Fare']


## Étape 6 : Pipeline pour la variable binaire `Sex`

La colonne `Sex` contient du texte avec seulement **deux catégories possibles** : `'male'` ou `'female'`.

Pour la convertir en nombre sans créer de colonne supplémentaire inutile :
1. **`SimpleImputer(strategy='most_frequent')`** : Par sécurité, si une valeur manquante apparaissait, elle serait remplacée par la catégorie la plus fréquente (`'male'`).

2. **`OrdinalEncoder(categories=[['male', 'female']])`** : Convertit directement sur une seule colonne :
   - `'male'` $\to 0.0$
   - `'female'` $\to 1.0$


In [13]:
colonne_sexe = ["Sex"]

pipeline_sexe = Pipeline([
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OrdinalEncoder(categories=[["male", "female"]]))
])

print("Pipeline binaire configuré pour :", colonne_sexe)


Pipeline binaire configuré pour : ['Sex']


## Étape 7 : Pipeline pour les variables catégorielles à plusieurs modalités (`Pclass`, `Embarked`)

Nous avons deux variables catégorielles comportant chacune **3 catégories** :
- **`Pclass`** : Classe du billet (`1`, `2` ou `3`).
- **`Embarked`** : Port d'embarquement (`'C'` = Cherbourg, `'Q'` = Queenstown, `'S'` = Southampton), qui contient 2 valeurs manquantes.

### Pourquoi utiliser `OneHotEncoder(drop='first')` ?
- Pour `Embarked`, il n'existe aucun ordre entre Cherbourg, Queenstown et Southampton. Si on les codait par `0, 1, 2`, le SVM croirait que Southampton (`2`) est deux fois plus grand que Queenstown (`1`).


- Le **One-Hot Encoding** crée des colonnes binaires (`0` ou `1`) indépendantes :
  - Pour `Embarked` (modalités `'C'`, `'Q'`, `'S'`), l'option **`drop='first'`** supprime la première modalité (`'C'`) et conserve **2 colonnes** : `Embarked_Q` et `Embarked_S`.
  - *Pourquoi supprimer la première colonne avec `drop='first'` ?* Parce que si un passager a `Embarked_Q = 0` et `Embarked_S = 0`, le modèle sait déjà à 100 % qu'il a embarqué à Cherbourg (`'C'`) ! Garder la 3ème colonne créerait une redondance mathématique exacte (appelée *multicolinéarité*).
  - De même pour `Pclass` (`1`, `2`, `3`), `OneHotEncoder(drop='first')` crée **2 colonnes** : `Pclass_2` et `Pclass_3` (la 1ère classe étant la référence quand les deux valent `0`). Cela permet au SVM de donner un poids totalement libre à chaque classe sociale.


In [14]:
colonnes_categorielles = ["Pclass", "Embarked"]

pipeline_categoriel = Pipeline([
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

print("Pipeline catégoriel One-Hot configuré pour :", colonnes_categorielles)


Pipeline catégoriel One-Hot configuré pour : ['Pclass', 'Embarked']


## Étape 8 : Assemblage de tous les traitements avec `ColumnTransformer` et `Pipeline`

Jusqu'ici, nous avons défini 4 petits pipelines spécialisés selon la nature des colonnes.
L'outil **`ColumnTransformer`** de `scikit-learn` joue le rôle d'aiguilleur : il envoie chaque groupe de colonnes vers son pipeline dédié, puis recolle toutes les colonnes transformées côte à côte dans une matrice finale propre.


In [15]:
# Regroupement des 4 branches de traitement dans le ColumnTransformer
pretraitement = ColumnTransformer(
    transformers=[
        ("numerique", pipeline_numerique, colonnes_numeriques_normales),
        ("fare", pipeline_fare, colonne_fare),
        ("sexe", pipeline_sexe, colonne_sexe),
        ("categoriel", pipeline_categoriel, colonnes_categorielles)
    ]
)

# Encapsulation dans le pipeline final de prétraitement
pipeline_final = Pipeline([
    ("pretraitement", pretraitement)
])

# Affichage de la structure du pipeline
pipeline_final


,steps,"[('pretraitement', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numerique', ...), ('fare', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Étape 9 : Exécution du prétraitement et vérification des données transformées

Nous appliquons maintenant notre `pipeline_final` en respectant la distinction fondamentale entre `fit_transform` et `transform` :

1. **Sur `X_train` : `pipeline_final.fit_transform(X_train)`**
   - **`fit`** : Le pipeline observe `X_train` et mémorise les paramètres statistiques (médianes de `Age`, `SibSp`, `Parch`, `Fare`, moyennes et écarts-types pour `StandardScaler`, modalités les plus fréquentes pour `Sex`, `Pclass`, `Embarked`).
   - **`transform`** : Il applique immédiatement ces transformations à `X_train`.

<br>

2. **Sur `X_test` : `pipeline_final.transform(X_test)`**
   - On appelle **uniquement `transform`** (jamais `fit` sur le jeu de test !). Le jeu de test est nettoyé et mis à l'échelle en utilisant exactement les médianes, moyennes et écarts-types appris sur le jeu d'entraînement.


In [16]:
# 1. Apprentissage des statistiques et transformation sur le jeu d'entraînement
X_train_prepare = pipeline_final.fit_transform(X_train)

# 2. Application stricte des mêmes transformations sur le jeu de test
X_test_prepare = pipeline_final.transform(X_test)

print("Dimensions de X_train_prepare :", X_train_prepare.shape)
print("Dimensions de X_test_prepare  :", X_test_prepare.shape)


Dimensions de X_train_prepare : (1047, 9)
Dimensions de X_test_prepare  : (262, 9)


### Inspection visuelle et contrôle qualité du tableau final prêt pour le SVM

En sortie de `scikit-learn`, `X_train_prepare` et `X_test_prepare` sont des tableaux numériques `numpy` (*arrays*).
Pour bien comprendre à quoi correspondent les **9 colonnes finales**, reconstruisons un `DataFrame` `pandas` avec les noms des colonnes générées et vérifions :
1. Qu'il ne reste **aucune valeur manquante (`NaN`)**.
2. Que toutes les colonnes sont **100 % numériques**.
3. Que les variables numériques ont bien une **moyenne proche de 0** et un **écart-type égal à 1** sur `X_train`.


In [19]:
# Récupération automatique des noms des 9 colonnes après transformation
noms_colonnes_finales = pipeline_final.named_steps["pretraitement"].get_feature_names_out()

# Conversion en DataFrame pandas pour une lecture claire
df_train_prepare = pd.DataFrame(X_train_prepare, columns=noms_colonnes_finales, index=X_train.index)
df_test_prepare = pd.DataFrame(X_test_prepare, columns=noms_colonnes_finales, index=X_test.index)

print("Liste des 9 variables finales envoyées au SVM :")
for i, col in enumerate(noms_colonnes_finales, 1):
    print(f"  {i}. {col}")

print("\nAperçu des 5 premières lignes de X_train_prepare")
display(df_train_prepare.head())

print("\nContrôle de sécurité avant le SVM")
print("Nombre total de valeurs manquantes (NaN) dans X_train_prepare :", np.isnan(X_train_prepare).sum())
print("Nombre total de valeurs manquantes (NaN) dans X_test_prepare  :", np.isnan(X_test_prepare).sum())

print("\nStatistiques descriptives de X_train_prepare (Moyenne ~0 et Écart-type ~1 sur les variables numériques)")
display(df_train_prepare.describe().T[["count", "mean", "std", "min", "50%", "max"]].round(3))


Liste des 9 variables finales envoyées au SVM :
  1. numerique__Age
  2. numerique__SibSp
  3. numerique__Parch
  4. fare__Fare
  5. sexe__Sex
  6. categoriel__Pclass_2
  7. categoriel__Pclass_3
  8. categoriel__Embarked_Q
  9. categoriel__Embarked_S

Aperçu des 5 premières lignes de X_train_prepare


,numerique__Age,numerique__SibSp,numerique__Parch,fare__Fare,sexe__Sex,categoriel__Pclass_2,categoriel__Pclass_3,categoriel__Embarked_Q,categoriel__Embarked_S
1096,-0.112493,-0.475419,-0.429878,0.334962,0.0,0.0,0.0,0.0,0.0
585,-0.889279,-0.475419,1.831843,1.460834,1.0,0.0,0.0,0.0,1.0
491,-0.656243,-0.475419,-0.429878,-0.864462,0.0,0.0,1.0,0.0,1.0
1127,2.683936,0.538356,-0.429878,1.403615,0.0,0.0,0.0,0.0,0.0
831,-2.223020,0.538356,0.700982,0.025858,0.0,1.0,0.0,0.0,1.0



Contrôle de sécurité avant le SVM
Nombre total de valeurs manquantes (NaN) dans X_train_prepare : 0
Nombre total de valeurs manquantes (NaN) dans X_test_prepare  : 0

Statistiques descriptives de X_train_prepare (Moyenne ~0 et Écart-type ~1 sur les variables numériques)


,count,mean,std,min,50%,max
numerique__Age,1047.0,-0.000,1.000,-2.262,-0.112,3.927
numerique__SibSp,1047.0,-0.000,1.000,-0.475,-0.475,7.635
numerique__Parch,1047.0,0.000,1.000,-0.430,-0.430,9.748
fare__Fare,1047.0,0.000,1.000,-3.017,-0.264,3.348
sexe__Sex,1047.0,0.355,0.479,0.000,0.000,1.000
categoriel__Pclass_2,1047.0,0.213,0.410,0.000,0.000,1.000
categoriel__Pclass_3,1047.0,0.545,0.498,0.000,1.000,1.000
categoriel__Embarked_Q,1047.0,0.092,0.289,0.000,0.000,1.000
categoriel__Embarked_S,1047.0,0.708,0.455,0.000,1.000,1.000


## Étape 10 : Exportation des ensembles `Train` et `Test` au format CSV

Afin de pouvoir utiliser directement nos données nettoyées lors de l'étape de modélisation du SVM sans avoir à relancer le prétraitement, nous sauvegardons les deux ensembles dans le dossier `dataset/` :

- **`dataset/train.csv`** : Les **1 047 passagers** d'entraînement (les 9 variables explicatives préparées + la colonne cible `Survived`).

- **`dataset/test.csv`** : Les **262 passagers** de test (les 9 variables explicatives préparées + la colonne cible `Survived`).

Nous utilisons l'argument `index=False` dans `.to_csv()` pour éviter d'ajouter une colonne inutile contenant les anciens numéros de lignes.


In [20]:
# Assemblage des variables explicatives préparées (X) avec la variable cible (y)
train_final_df = df_train_prepare.copy()
train_final_df["Survived"] = y_train.values

test_final_df = df_test_prepare.copy()
test_final_df["Survived"] = y_test.values

# Exportation au format CSV dans le dossier dataset/
chemin_train = "dataset/train.csv"
chemin_test = "dataset/test.csv"

train_final_df.to_csv(chemin_train, index=False)
test_final_df.to_csv(chemin_test, index=False)

print(f"Fichier d'entraînement exporté : {chemin_train} ({train_final_df.shape[0]} lignes, {train_final_df.shape[1]} colonnes)")
print(f"Fichier de test exporté        : {chemin_test} ({test_final_df.shape[0]} lignes, {test_final_df.shape[1]} colonnes)")

print("\nAperçu du fichier dataset/train.csv")
display(train_final_df.head(3))


Fichier d'entraînement exporté : dataset/train.csv (1047 lignes, 10 colonnes)
Fichier de test exporté        : dataset/test.csv (262 lignes, 10 colonnes)

Aperçu du fichier dataset/train.csv


,numerique__Age,numerique__SibSp,numerique__Parch,fare__Fare,sexe__Sex,categoriel__Pclass_2,categoriel__Pclass_3,categoriel__Embarked_Q,categoriel__Embarked_S,Survived
1096,-0.112493,-0.475419,-0.429878,0.334962,0.0,0.0,0.0,0.0,0.0,1
585,-0.889279,-0.475419,1.831843,1.460834,1.0,0.0,0.0,0.0,1.0,1
491,-0.656243,-0.475419,-0.429878,-0.864462,0.0,0.0,1.0,0.0,1.0,0


## Bilan du prétraitement

Nos données sont désormais prêtes pour l'entraînement d'un modèle SVM :
- Les **15 colonnes brutes** initiales ont été épurées des fuites de données (`boat`, `body`) et des colonnes bruitées ou identifiantes (`PassengerId`, `Name`, `Ticket`, `Cabin`, `home.dest`).
- La cible `Survived` a été isolée dans `y_train` (1 047 passagers) et `y_test` (262 passagers) via un découpage stratifié (80 % / 20 %).
- Les **7 variables explicatives retenues** (`Age`, `SibSp`, `Parch`, `Fare`, `Sex`, `Pclass`, `Embarked`) ont été transformées sans aucune fuite de données en **9 colonnes numériques propres, sans aucun `NaN`, et parfaitement mises à l'échelle**.
- Les deux jeux de données finaux ont été sauvegardés dans `dataset/train.csv` et `dataset/test.csv`.
